# Дообучение детектора зданий (geoqc)

Ноутбук запускается локально и сам выбирает устройство: CUDA, Apple MPS или CPU.

Что делает ноутбук:

1. определяет устройство, фиксирует seed;
2. готовит датасет `data/interim/dataset` из своей разметки (`geoqc cv export-dataset`, D-021): тайлы `train` — обучение,
   тайлы `val` — holdout для ранней остановки и подбора порога;
3. дообучает YOLOv8-seg (Ultralytics): AMP, кэширование, чекпоинт каждой эпохи, resume, ранняя остановка, лог в CSV;
4. экспортирует лучшие веса в `models/`, считает sha256 → `model_version`, обновляет `config/cv.yaml` и `models/MODEL_CARD.md`;
5. подбирает порог уверенности на holdout (`threshold-sweep`).

Разметка тестового сплита в ноутбуке **не используется**: ни для обучения, ни для валидации, ни для подбора порога.
Рабочая модель (train + AIRS, D-024) обучена скриптом `scripts/experiments/airs_night.py` по той же схеме.
Порядок запуска и ожидаемое время — в `notebooks/README.md`.

## 1. Устройство и seed

In [ ]:
# Pick the device (cuda > mps > cpu), fix seeds.
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
os.environ.setdefault("PYTHONHASHSEED", str(SEED))

print(f"device={DEVICE} torch={torch.__version__} python={platform.python_version()}")
print(f"cuda={torch.cuda.is_available()} mps={torch.backends.mps.is_available()}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Пути и конфигурация

Корень репозитория ищется вверх от текущего каталога по `pyproject.toml`. Пакет `geoqc` импортируется из `src/`,
конфиг читается `geoqc.config.load_config()`.

In [ ]:
# Resolve the repository root and load config/*.yaml through geoqc.config.
import yaml  # noqa: E402

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
os.environ["GEOQC_CONFIG_DIR"] = str(REPO_ROOT / "config")

from geoqc.config import load_config  # noqa: E402

CFG = load_config(REPO_ROOT / "config")
CV = CFG["cv"]
DATASET_DIR = REPO_ROOT / CV["dataset"]["out_dir"]
MODELS_DIR = REPO_ROOT / "models"
RUNS_DIR = MODELS_DIR / "runs"  # models/* is git-ignored except MODEL_CARD.md
REPORTS_DIR = REPO_ROOT / "reports"
for d in (MODELS_DIR, RUNS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)
print("backend =", CV["backend"], "| model_version =", CV.get("model_version"))
print(yaml.safe_dump({"dataset": CV["dataset"], "vectorize": CV["vectorize"]}, sort_keys=False))

## 3. Данные: своя разметка

`export_dataset` (D-021) берёт тайлы `train` с `data/labels/train/buildings.geojson` в `images/train`, тайлы `val` со своей
разметкой — в `images/holdout` (ранняя остановка, подбор порога); разметка `test` не читается. Пишет YOLO-seg
`labels/*.txt`, бинарные `masks/*.png`, `data.yaml`, `split.csv`, `meta.json`. Если датасет уже экспортирован, экспорт
пропускается; `path` в `data.yaml` переписывается под текущую машину.

Внешние данные (SpaceNet 2, footprints Microsoft, D-019/D-020) пробовались и не помогли; их загрузчики удалены (D-027).
AIRS подмешивается скриптом `scripts/experiments/airs_night.py`.

In [ ]:
# Training material: our verified labels (train -> training, val -> holdout).
from geoqc.cv.dataset import export_dataset, read_holdout_tile_ids  # noqa: E402

DATA_YAML = DATASET_DIR / "data.yaml"
TRAIN_LABELS = REPO_ROOT / "data" / "labels" / "train" / "buildings.geojson"
if not DATA_YAML.exists() and not TRAIN_LABELS.exists():
    raise RuntimeError(
        f"{TRAIN_LABELS.relative_to(REPO_ROOT)} is missing: finish the labels in QGIS and run "
        "`geoqc annotation finalize --split train`"
    )
if not DATA_YAML.exists():
    summary = export_dataset(CV, CFG)
    print(summary.to_dict())
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
data_cfg["path"] = str(DATASET_DIR)  # the dataset may come from another machine
data_cfg["train"] = "images/train"
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")
N_TRAIN = len(list((DATASET_DIR / "images" / "train").glob("*.png")))
N_HOLDOUT = len(list((DATASET_DIR / "images" / "holdout").glob("*.png")))
print(f"train tiles={N_TRAIN} holdout tiles={N_HOLDOUT}")
print("holdout ids:", read_holdout_tile_ids(DATASET_DIR)[:20])
if N_TRAIN == 0:
    raise RuntimeError("no training tiles: verify the labels in QGIS and re-run `geoqc cv export-dataset`")

## 4. Параметры обучения

Дообучение YOLOv8-seg от исходного чекпоинта `BASE_WEIGHTS` (`models/yolov8m-building-seg.pt`, при его
отсутствии скачивается с HF или берётся fallback `yolov8m-seg.pt`). Не от `config/cv.yaml → weights.path`: после экспорта там
лежат веса прошлого прогона, и полный прогон после smoke-теста стартовал бы с них. Продолжить дообучение готовых весов —
указать их в `BASE_WEIGHTS` явно. `RESUME = True` продолжает
прерванный прогон с `last.pt`.

`SMOKE_TEST = True` делает 2 эпохи, чтобы проверить весь путь до экспорта весов: Ultralytics отводит на прогрев lr
`min(warmup_epochs, epochs − 1)` эпох, и при одной эпохе прогрева нет.

YOLO дообучается с `optimizer="AdamW"`, `lr0 = YOLO_LR0` (2e-4) и `warmup_bias_lr = 0`. Режим `optimizer="auto"` для ~60
итераций на эпоху выбирает AdamW с lr 2e-3, и первые шаги стирают уже обученный на зданиях чекпоинт. `warmup_bias_lr = 0,1`
по умолчанию подобран для SGD: с AdamW он за пару шагов ломает голову масок (smoke-прогон на val: mAP50 масок 0,48 → 0,003).
Замер на SpaceNet (2 эпохи, holdout, mAP50 масок): исходные веса 0,56; auto 0,29; AdamW 2e-4 — 0,67; то же с
`warmup_bias_lr = 0` — 0,73. Это проверка режима обучения, а не метрика работы.

In [ ]:
# Training hyper-parameters (edit here only).
RUN_NAME = "finetune_yolov8m_seg_ours"
SMOKE_TEST = False
EPOCHS = 2 if SMOKE_TEST else 60  # 1 epoch = no lr warmup in ultralytics
PATIENCE = 12
IMGSZ = int(CV.get("imgsz", 512))
BATCH = {"cuda": 8, "mps": 4, "cpu": 2}[DEVICE]
WORKERS = 0
CACHE = "disk"  # Ultralytics: 'ram' | 'disk' | False
RESUME = False
YOLO_LR0 = 2e-4  # AdamW; "auto" picks 2e-3 on small datasets and wipes out the checkpoint
BASE_WEIGHTS = MODELS_DIR / "yolov8m-building-seg.pt"  # pretrained checkpoint, not the last export
RUN_DIR = RUNS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"epochs={EPOCHS} batch={BATCH} imgsz={IMGSZ} device={DEVICE} resume={RESUME}")

## 5. Обучение YOLOv8-seg (Ultralytics)

`model.train(...)` c `amp=True` (Ultralytics сам отключает AMP вне CUDA), `cache`, `workers`, `imgsz=512`, `patience`
(ранняя остановка), `save_period=1` (чекпоинт каждой эпохи), `seed`/`deterministic`. Лог метрик — `RUN_DIR/results.csv`.
Resume: `YOLO(last.pt).train(resume=True)`. Валидация в процессе обучения идёт по `images/holdout` (`val:` в `data.yaml`).

In [ ]:
# YOLOv8-seg fine-tuning with Ultralytics.
from ultralytics import YOLO  # noqa: E402

from geoqc.cv.weights import fetch_weights  # noqa: E402

BEST_WEIGHTS: Path | None = None

base_weights = BASE_WEIGHTS
if not base_weights.exists():  # HF checkpoint or COCO fallback (see MODEL_CARD)
    base_cfg = {
        **CV,
        "weights": {**CV["weights"], "path": str(BASE_WEIGHTS), "is_fallback": False},
    }
    info = fetch_weights(base_cfg, update_files=False)
    base_weights = REPO_ROOT / info.path
    print("fetched", info.model_version, "fallback" if info.is_fallback else "")
last_pt = RUN_DIR / "weights" / "last.pt"
if RESUME and last_pt.exists():
    model = YOLO(str(last_pt))
    results = model.train(resume=True)
else:
    model = YOLO(str(base_weights))
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        amp=True,
        optimizer="AdamW",
        lr0=YOLO_LR0,
        warmup_bias_lr=0.0,  # default 0.1 is an SGD value: with AdamW it wipes the mask head
        cache=CACHE,
        workers=WORKERS,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        deterministic=True,
        save_period=1,
        plots=False,
        verbose=False,
    )
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("best weights:", BEST_WEIGHTS, BEST_WEIGHTS.exists())
print("metrics log:", RUN_DIR / "results.csv")

## 6. Экспорт весов → `model_version`, `config/cv.yaml`, `MODEL_CARD.md`

Лучшие веса копируются в `models/<model_name>.pt`, считается sha256, `model_version = "<model_name>-<sha256[:8]>"`
(контракт §3.6). В `config/cv.yaml` обновляются `backend`, `model_name`, путь к весам, sha256 и `model_version`;
в `models/MODEL_CARD.md` перезаписывается автоблок с гиперпараметрами прогона.

In [ ]:
# Export best weights, compute sha256 -> model_version, update config/cv.yaml and MODEL_CARD.md.
import datetime as dt  # noqa: E402
import shutil  # noqa: E402

from geoqc.cv.weights import (  # noqa: E402
    WeightsInfo,
    model_version_for,
    sha256_of,
    update_cv_config,
    update_model_card,
)

if BEST_WEIGHTS is None or not Path(BEST_WEIGHTS).exists():
    raise RuntimeError("no trained weights found; run the training cell first")
MODEL_NAME, BACKEND = "yolov8m-building-seg-ft", "ultralytics"
dest = MODELS_DIR / f"{MODEL_NAME}.pt"
shutil.copyfile(BEST_WEIGHTS, dest)
SHA = sha256_of(dest)
MODEL_VERSION = model_version_for(MODEL_NAME, SHA)
rel_dest = dest.relative_to(REPO_ROOT).as_posix()
update_cv_config(
    {
        "backend": BACKEND,
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "weights.path": rel_dest,
        "weights.sha256": SHA,
        "weights.is_fallback": False,
    },
    REPO_ROOT / "config" / "cv.yaml",
)
info = WeightsInfo(
    path=rel_dest,
    sha256=SHA,
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
    source_url=f"fine-tuned in notebooks/train_finetune.ipynb from {CV['weights'].get('hf_repo') or CV['weights'].get('fallback_asset')}",
    license="AGPL-3.0 (derivative of Ultralytics weights)",
    is_fallback=False,
    backend=BACKEND,
    fetched_at=dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
)
extra = {
    "fine-tune": f"yolo: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, patience={PATIENCE}, seed={SEED}, device={DEVICE}, amp=True",
    "train data": f"our labels: {N_TRAIN} train tiles; holdout {N_HOLDOUT} val tiles",
    "run dir": RUN_DIR.relative_to(REPO_ROOT).as_posix(),
}
update_model_card(info, REPO_ROOT / "models" / "MODEL_CARD.md", extra=extra)
print("model_version =", MODEL_VERSION)
print("weights ->", dest)

## 7. Подбор порога уверенности на holdout (val)

`run_sweep(holdout_only=True)` запускает инференс новыми весами на holdout-тайлах при `sweep.conf_floor`, для каждого порога
из `sweep.thresholds` заново склеивает/фильтрует полигоны и считает P/R/F1 при IoU ≥ 0,5 (формулы `CONTRACTS §4`;
`geoqc.eval.detection_metrics`, если модуль доступен). Лучший порог записывается в `config/cv.yaml → conf_threshold`,
отчёт — `reports/threshold_sweep_val.json`. Тестовый сплит не участвует.

In [ ]:
# Threshold sweep on the holdout subset of val with the freshly exported weights.
from geoqc.cv.sweep import run_sweep  # noqa: E402

CFG = load_config(REPO_ROOT / "config")  # reload: weights/backend were just rewritten
report = run_sweep(CFG["cv"], CFG, holdout_only=True, device=DEVICE, progress=False)
print(
    "best:",
    report["best"],
    "| n_gt:",
    report["n_gt"],
    "| tiles:",
    report["n_tiles"],
    "| impl:",
    report["metrics_impl"],
)
for row in report["thresholds"]:
    print(
        f"conf={row['conf']:.2f} n={row['n_det']:4d} P={row['precision']:.3f} R={row['recall']:.3f} F1={row['f1']:.3f}"
    )
print("report:", report["report_path"])

## 8. Что дальше

* `geoqc cv predict --split all --to-db` → `data/interim/detections.geojson` + `buildings_det`;
  затем `geoqc matching run --model-version <model_version>` и `geoqc eval detection --split val` (гейт G3 ≥ 0,70).
* Тестовый сплит (`geoqc eval detection --split test`, гейт G4) оценивается один раз — после фиксации весов и порога.